# Run Sandbox Extra-Keypoints Retargeting Experiments


## Locate Project Files

This cell finds the extra-keypoints sandbox script from the current notebook working directory and sets the subprocess working directory to the outer `holosoma_retargeting/` package root, matching the command-line workflow used by the examples.


In [11]:
from __future__ import annotations

from pathlib import Path
import shlex
import subprocess

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def find_extra_keypoints_sandbox_script(start: Path) -> Path:
    candidates = []
    for root in [start, *start.parents]:
        candidates.extend(
            [
                root / "holosoma_retargeting" / "holosoma_retargeting" / "examples" / "robot_retarget_sandbox_extra_keypoints_experimental.py",
                root / "holosoma_retargeting" / "examples" / "robot_retarget_sandbox_extra_keypoints_experimental.py",
            ]
        )
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate robot_retarget_sandbox_extra_keypoints_experimental.py from the current working directory")

NOTEBOOK_CWD = Path.cwd().resolve()
SCRIPT_PATH = find_extra_keypoints_sandbox_script(NOTEBOOK_CWD)
PACKAGE_ROOT = SCRIPT_PATH.parents[1]
RUN_WORKDIR = PACKAGE_ROOT
SCRIPT_REL = SCRIPT_PATH.relative_to(RUN_WORKDIR)

print("notebook cwd:", NOTEBOOK_CWD)
print("script:", SCRIPT_PATH)
print("run cwd:", RUN_WORKDIR)
print("script rel:", SCRIPT_REL)


notebook cwd: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/examples
script: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/examples/robot_retarget_sandbox_extra_keypoints_experimental.py
run cwd: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting
script rel: examples/robot_retarget_sandbox_extra_keypoints_experimental.py


## Configure One Experiment

这一块只是在 notebook 里配置一次 extra-keypoints sandbox retargeting 实验，以及后面要不要画诊断图。

- `TASK_TYPE` / `ROBOT` / `TASK_NAME` / `DATA_FORMAT` / `DATA_PATH` 决定跑哪条数据、用哪个机器人、怎么读输入。
- `SAVE_DIR` / `AUGMENTATION` 决定结果写到哪里，以及输出文件名用 `_original` 还是 `_augmented` 后缀。
- `EXTRA_ARGS` 会原样追加到 Tyro 命令行里，用来临时覆盖 retargeter 参数。
- `RUN_EXPERIMENT` 是真正执行实验的开关；`False` 只构造命令和检查已有输出，`True` 会重新运行并覆盖同名输出。
- `PLOT_FRAME` / `PLOT_INNER` / `PLOTTED_HESSIAN_COMPONENTS` 只控制本 notebook 的图，不改变 retargeting 结果。

`SAVE_DIR = None` 时使用 sandbox 默认输出目录：`demo_results_total/sandbox_extra_keypoints_experimental/`。


In [12]:
# 子进程使用的 Conda 环境。
CONDA_ENV = "robot"

# 实验身份：决定输入序列、机器人，以及预期输出文件名。
TASK_TYPE = "object_interaction"  # 可选："robot_only", "object_interaction", "climbing"
ROBOT = "g1"
TASK_NAME = "sub3_largebox_003"
DATA_FORMAT = None  # None 表示让 robot_retarget_sandbox_extra_keypoints_experimental.py 使用该任务的默认格式。
DATA_PATH = Path("demo_data/OMOMO_new")

# 输出控制：SAVE_DIR=None 使用 sandbox 默认目录；AUGMENTATION 改变输出文件后缀。
SAVE_DIR = None
AUGMENTATION = False

# 额外 Tyro CLI 参数，会追加在标准参数后面。
# 例子：EXTRA_ARGS = ["--retargeter.step-size", "0.15", "--retargeter.debug"]
EXTRA_ARGS: list[str] = []

# 执行开关：False 只打印/构造命令；True 会实际运行并覆盖预期输出路径。
RUN_EXPERIMENT = True

# 画图开关：只影响本 notebook 的图，不影响实验运行。
PLOT_FRAME = True  # 每帧一条 Hessian 曲线，取该 frame 的最后一个 inner iteration。
PLOT_INNER = True  # 所有 inner-iteration Hessian 记录，按 frame 和 inner iteration 排序。

# 要画的 Hessian component；需要时可加 "lap_j_only", "lap_robot_rows", "lap_object_rows"。
PLOTTED_HESSIAN_COMPONENTS = ["lap"]


## Build Command And Expected Output Path

The output path mirrors `determine_output_path()` and the extra-keypoints sandbox default save directory in `robot_retarget_sandbox_extra_keypoints_experimental.py`.


In [13]:
DEFAULT_RESULTS_ROOT = PACKAGE_ROOT / "demo_results_total"
DEFAULT_SAVE_DIRS = {
    "robot_only": DEFAULT_RESULTS_ROOT / "sandbox_extra_keypoints_experimental" / "{robot}" / "robot_only" / "omomo",
    "object_interaction": DEFAULT_RESULTS_ROOT / "sandbox_extra_keypoints_experimental" / "{robot}" / "object_interaction" / "omomo",
    "climbing": DEFAULT_RESULTS_ROOT / "sandbox_extra_keypoints_experimental" / "{robot}" / "climbing" / "mocap_climb",
}


def resolve_run_path(path: Path | str | None) -> Path | None:
    if path is None:
        return None
    path = Path(path)
    return path if path.is_absolute() else RUN_WORKDIR / path


def default_save_dir(task_type: str, robot: str) -> Path:
    return Path(str(DEFAULT_SAVE_DIRS[task_type]).format(robot=robot))


def expected_result_path(task_type: str, save_dir: Path, task_name: str, augmentation: bool) -> Path:
    if task_type == "robot_only":
        return save_dir / f"{task_name}.npz"
    suffix = "_augmented" if augmentation else "_original"
    return save_dir / f"{task_name}{suffix}.npz"

resolved_data_path = resolve_run_path(DATA_PATH)
resolved_save_dir = resolve_run_path(SAVE_DIR) if SAVE_DIR is not None else default_save_dir(TASK_TYPE, ROBOT)
result_path = expected_result_path(TASK_TYPE, resolved_save_dir, TASK_NAME, AUGMENTATION)

cmd = [
    "conda",
    "run",
    "-n",
    CONDA_ENV,
    "python",
    str(SCRIPT_REL),
    "--task-type",
    TASK_TYPE,
    "--robot",
    ROBOT,
    "--task-name",
    TASK_NAME,
    "--data-path",
    str(Path(DATA_PATH)),
]
if DATA_FORMAT is not None:
    cmd.extend(["--data-format", DATA_FORMAT])
if SAVE_DIR is not None:
    cmd.extend(["--save-dir", str(Path(SAVE_DIR))])
if AUGMENTATION:
    cmd.append("--augmentation")
cmd.extend(EXTRA_ARGS)

print("command:")
print(shlex.join(cmd))
print("run cwd:", RUN_WORKDIR)
print("expected result:", result_path)


command:
conda run -n robot python examples/robot_retarget_sandbox_extra_keypoints_experimental.py --task-type object_interaction --robot g1 --task-name sub3_largebox_003 --data-path demo_data/OMOMO_new
run cwd: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting
expected result: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/sandbox_extra_keypoints_experimental/g1/object_interaction/omomo/sub3_largebox_003_original.npz


## Run The Experiment

Set `RUN_EXPERIMENT = True` in the configuration cell, then run this cell. Output is streamed so long retargeting runs show progress in the notebook.


In [14]:
if RUN_EXPERIMENT:
    process = subprocess.Popen(
        cmd,
        cwd=RUN_WORKDIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f"Extra-keypoints sandbox retargeting failed with exit code {returncode}")
else:
    print("RUN_EXPERIMENT is False; command was not executed.")
    print("Set RUN_EXPERIMENT = True in the configuration cell to run it.")

print("expected result exists:", result_path.exists())


Loading and sampling object mesh...
Loading robot model from:  models/g1/g1_29dof_w_largebox.xml

Starting motion retargeting for 196 frames...
Saving results to path: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/sandbox_extra_keypoints_experimental/g1/object_interaction/omomo/sub3_largebox_003_original.npz
Saving Hessian component diagnostics to path: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/sandbox_extra_keypoints_experimental/g1/object_interaction/omomo/sub3_largebox_003_original_hessian_components.npz

INFO: Task: sub3_largebox_003, Type: object_interaction, Format: smplh
INFO: Data path: demo_data/OMOMO_new, Save dir: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/sandbox_extra_keypoints_experimental/g1/object_interaction/omomo
INFO: Loading motion data for task: sub3_largebox_0

## Load Result

Run this after the experiment has produced the expected main result `.npz` file. Hessian component matrices are stored in a sidecar `.npz` next to the main result and loaded separately below.

In [15]:
if not result_path.exists():
    raise FileNotFoundError(f"Result file does not exist yet: {result_path}")

data = np.load(result_path)
if "hessian_components_file" not in data.files:
    raise KeyError("Result file does not contain hessian_components_file; rerun the sandbox experiment.")

hessian_components_path = result_path.parent / str(data["hessian_components_file"].item())
if not hessian_components_path.exists():
    raise FileNotFoundError(f"Missing Hessian component sidecar: {hessian_components_path}")

hessian_data = np.load(hessian_components_path)
print("loaded result:", result_path)
print("result fields:")
for key in data.files:
    value = data[key]
    print(f"  {key:40s} shape={value.shape} dtype={value.dtype}")

print("loaded Hessian components:", hessian_components_path)
print("Hessian component fields:")
for key in hessian_data.files:
    value = hessian_data[key]
    print(f"  {key:40s} shape={value.shape} dtype={value.dtype}")


loaded result: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/sandbox_extra_keypoints_experimental/g1/object_interaction/omomo/sub3_largebox_003_original.npz
result fields:
  qpos                                     shape=(196, 43) dtype=float64
  human_joints                             shape=(196, 52, 3) dtype=float32
  fps                                      shape=() dtype=int64
  cost                                     shape=() dtype=float64
  original_lap_cost                        shape=(196,) dtype=float64
  original_smooth_cost                     shape=(196,) dtype=float64
  original_lap_smooth_cost                 shape=(196,) dtype=float64
  hessian_components_file                  shape=() dtype=<U49
loaded Hessian components: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/sandbox_extra_keypoints_experimental/g1/object_interaction/omomo/sub3

## Check Hessian Component Fields

These checks validate the sidecar Hessian component matrix file produced by `interaction_mesh_retargeter_sandbox_extra_keypoints_experimental.py`.

In [16]:
HESSIAN_COMPONENT_FIELDS = [
    "source_result_file",
    "hessian_frame",
    "hessian_inner_iter",
    "hessian_component_names",
    "hessian_component_matrices",
    "frame_hessian_component_matrices",
]
missing = [key for key in HESSIAN_COMPONENT_FIELDS if key not in hessian_data.files]
if missing:
    raise KeyError(f"Missing Hessian component diagnostic fields: {missing}")

num_frames = data["qpos"].shape[0]
hessian_frames = hessian_data["hessian_frame"].astype(int)
hessian_inner_iters = hessian_data["hessian_inner_iter"].astype(int)
hessian_component_names = [str(name) for name in hessian_data["hessian_component_names"]]
hessian_component_matrices = np.asarray(hessian_data["hessian_component_matrices"], dtype=float)
frame_hessian_component_matrices = np.asarray(hessian_data["frame_hessian_component_matrices"], dtype=float)

expected_component_names = [
    "lap",
    "lap_j_only",
    "lap_robot_rows",
    "lap_object_rows",
    "nominal",
    "q_diag",
    "smooth",
    "total",
]
if hessian_component_names != expected_component_names:
    raise ValueError(
        f"Unexpected Hessian component order: {hessian_component_names}. "
        "Regenerate the extra-keypoints sandbox Hessian diagnostics."
    )

expected_inner_shape = (hessian_frames.size, len(expected_component_names))
if hessian_component_matrices.ndim != 4 or hessian_component_matrices.shape[:2] != expected_inner_shape:
    raise ValueError(
        "hessian_component_matrices shape mismatch: "
        f"shape={hessian_component_matrices.shape}, expected prefix={expected_inner_shape}"
    )
if hessian_component_matrices.shape[-2] != hessian_component_matrices.shape[-1]:
    raise ValueError(f"Hessian matrices must be square, got {hessian_component_matrices.shape[-2:]}")

nq_a = hessian_component_matrices.shape[-1]
expected_frame_shape = (num_frames, len(expected_component_names), nq_a, nq_a)
if frame_hessian_component_matrices.shape != expected_frame_shape:
    raise ValueError(
        "frame_hessian_component_matrices shape mismatch: "
        f"shape={frame_hessian_component_matrices.shape}, expected={expected_frame_shape}"
    )

component_index = {name: i for i, name in enumerate(hessian_component_names)}
print("num frames:", num_frames)
print("num inner-iteration Hessian records:", hessian_frames.size)
print("Hessian components:", hessian_component_names)
print("Hessian matrix dimension:", nq_a)


num frames: 196
num inner-iteration Hessian records: 1865
Hessian components: ['lap', 'lap_j_only', 'lap_robot_rows', 'lap_object_rows', 'nominal', 'q_diag', 'smooth', 'total']
Hessian matrix dimension: 36


## Plot Helpers

These helpers compute Hessian component eigenvalue diagnostics from the sidecar matrix arrays and draw each min/max eigenvalue curve with Plotly. `PLOTTED_HESSIAN_COMPONENTS` controls only the Hessian eigenvalue component selection; the right y-axis can still overlay the available original cost curves.

In [17]:
COST_FIELDS = [
    ("original_lap_cost", "original lap cost"),
    ("original_smooth_cost", "original smooth cost"),
    ("original_lap_smooth_cost", "original lap+smooth cost"),
]

HESSIAN_LABELS = {
    "lap": "Laplacian full",
    "lap_j_only": "J-only",
    "lap_robot_rows": "Laplacian robot rows",
    "lap_object_rows": "Laplacian object rows",
    "nominal": "Nominal tracking",
    "q_diag": "Q diagonal",
    "smooth": "Smoothness",
    "total": "Total",
}

HESSIAN_METRICS = [
    ("min_eig", "minimum eigenvalue"),
    ("max_eig", "maximum eigenvalue"),
]


def finite_for_plot(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    plotted = values.copy()
    plotted[~np.isfinite(plotted)] = np.nan
    return plotted


def available_cost_fields(data) -> list[tuple[str, str, np.ndarray]]:
    cost_series = []
    for field, label in COST_FIELDS:
        if field in data.files:
            cost_series.append((field, label, np.asarray(data[field], dtype=float).reshape(-1)))
        else:
            print(f"Skipping missing cost field: {field}")
    return cost_series


def hessian_component_extreme_eigenvalues(component_matrices: np.ndarray) -> dict[str, dict[str, np.ndarray]]:
    component_matrices = np.asarray(component_matrices, dtype=float)
    result = {
        name: {
            "min_eig": np.full(component_matrices.shape[0], np.nan, dtype=float),
            "max_eig": np.full(component_matrices.shape[0], np.nan, dtype=float),
        }
        for name in hessian_component_names
    }
    for record_idx in range(component_matrices.shape[0]):
        for component_name, component_idx in component_index.items():
            H = component_matrices[record_idx, component_idx]
            if np.all(np.isfinite(H)):
                eigvals = np.linalg.eigvalsh(0.5 * (H + H.T))
                result[component_name]["min_eig"][record_idx] = float(eigvals[0])
                result[component_name]["max_eig"][record_idx] = float(eigvals[-1])
    return result


def plot_metric_with_costs(
    *,
    x: np.ndarray,
    metric_values: np.ndarray,
    metric_label: str,
    title: str,
    x_label: str,
    cost_series: list[tuple[str, str, np.ndarray]],
    cost_indexer: np.ndarray | None = None,
    customdata: np.ndarray | None = None,
) -> None:
    x = np.asarray(x)
    metric_plot = finite_for_plot(metric_values)
    if not np.isfinite(metric_plot).any():
        print(f"Skipping {title}: no finite metric values.")
        return

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    hovertemplate = f"{x_label}=%{{x}}<br>{metric_label}=%{{y:.6e}}<extra></extra>"
    if customdata is not None:
        hovertemplate = (
            "record=%{x}<br>frame=%{customdata[0]}<br>inner_iter=%{customdata[1]}"
            f"<br>{metric_label}=%{{y:.6e}}<extra></extra>"
        )

    fig.add_trace(
        go.Scatter(
            x=x,
            y=metric_plot,
            mode="lines",
            name=metric_label,
            customdata=customdata,
            hovertemplate=hovertemplate,
            line={"width": 2.5},
        ),
        secondary_y=False,
    )

    for field, label, cost_values in cost_series:
        if cost_indexer is None:
            cost_x = np.arange(cost_values.size)
            cost_y = finite_for_plot(cost_values)
        else:
            if cost_indexer.size == 0 or cost_indexer.min() < 0 or cost_indexer.max() >= cost_values.size:
                print(f"Skipping {field} for {title}: cost index out of range.")
                continue
            cost_x = x
            cost_y = finite_for_plot(cost_values[cost_indexer])

        fig.add_trace(
            go.Scatter(
                x=cost_x,
                y=cost_y,
                mode="lines",
                name=label,
                opacity=0.65,
                line={"dash": "dot", "width": 1.5},
                hovertemplate=f"{x_label}=%{{x}}<br>{label}=%{{y:.6e}}<extra></extra>",
            ),
            secondary_y=True,
        )

    fig.update_layout(
        title=title,
        height=420,
        hovermode="x unified",
        legend_title_text="series",
        margin={"l": 70, "r": 70, "t": 70, "b": 55},
    )
    fig.update_xaxes(title_text=x_label)
    fig.update_yaxes(title_text=metric_label, type="linear", tickformat=".2e", secondary_y=False)
    fig.update_yaxes(title_text="cost", secondary_y=True)
    fig.show()


def plot_hessian_metric_grid(
    *,
    spectra: dict[str, dict[str, np.ndarray]],
    x: np.ndarray,
    title_prefix: str,
    x_label: str,
    cost_series: list[tuple[str, str, np.ndarray]],
    value_indexer: np.ndarray | None = None,
    cost_indexer: np.ndarray | None = None,
    customdata: np.ndarray | None = None,
) -> None:
    for component_name in PLOTTED_HESSIAN_COMPONENTS:
        if component_name not in spectra:
            print(f"Skipping unknown Hessian component: {component_name}")
            continue
        component_label = HESSIAN_LABELS.get(component_name, component_name)
        for metric_key, metric_name in HESSIAN_METRICS:
            metric_values = spectra[component_name][metric_key]
            if value_indexer is not None:
                metric_values = metric_values[value_indexer]
            plot_metric_with_costs(
                x=x,
                metric_values=metric_values,
                metric_label=f"{component_label} {metric_name}",
                title=f"{title_prefix} {component_label} Hessian {metric_name}",
                x_label=x_label,
                cost_series=cost_series,
                cost_indexer=cost_indexer,
                customdata=customdata,
            )


cost_series = available_cost_fields(data)
frame_hessian_spectra = hessian_component_extreme_eigenvalues(frame_hessian_component_matrices)
inner_hessian_spectra = hessian_component_extreme_eigenvalues(hessian_component_matrices)


## Per-Frame Hessian Eigenvalues With Costs

Uses `frame_hessian_component_matrices`, where each frame keeps the last inner iteration.


In [18]:
if PLOT_FRAME:
    plot_hessian_metric_grid(
        spectra=frame_hessian_spectra,
        x=np.arange(num_frames),
        title_prefix="Per-frame",
        x_label="frame",
        cost_series=cost_series,
    )
else:
    print("PLOT_FRAME is False; skipping per-frame Hessian plots.")


## Inner-Iteration Hessian Eigenvalues With Costs

Uses every raw Hessian component matrix record, sorted by `(frame, inner_iter)`. This section is controlled by `PLOT_INNER`.


In [19]:
if PLOT_INNER:
    if hessian_frames.size == 0:
        raise ValueError("No Hessian inner-iteration records are available.")

    inner_order = np.lexsort((hessian_inner_iters, hessian_frames))
    ordered_frames = hessian_frames[inner_order]
    ordered_inner_iters = hessian_inner_iters[inner_order]
    inner_x = np.arange(inner_order.size)
    inner_customdata = np.column_stack([ordered_frames, ordered_inner_iters])

    print(f"inner-iteration Hessian records: {inner_order.size}")
    print(f"frame range: {int(ordered_frames.min())}..{int(ordered_frames.max())}")
    print(f"inner iteration range: {int(ordered_inner_iters.min())}..{int(ordered_inner_iters.max())}")

    plot_hessian_metric_grid(
        spectra=inner_hessian_spectra,
        x=inner_x,
        title_prefix="Inner-expanded",
        x_label="record index sorted by frame and inner iteration",
        cost_series=cost_series,
        value_indexer=inner_order,
        cost_indexer=ordered_frames,
        customdata=inner_customdata,
    )
else:
    print("PLOT_INNER is False; skipping inner-iteration Hessian plots.")


inner-iteration Hessian records: 1865
frame range: 0..195
inner iteration range: 0..49
